# IPL Match Winner Prediction

A simple machine learning project that predicts the winner between two selected IPL teams.

## 1. Import libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay, roc_auc_score
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.preprocessing import OneHotEncoder

## 2. Load the dataset

In [ ]:
df = pd.read_csv("../data/ipl.csv")
print("Dataset shape:", df.shape)
df.head()

## 3. Inspect missing values

In [ ]:
df.isnull().sum()

## 4. Feature selection and cleaning

The model uses match information that is available around the start of a match. Historical team names are normalized so that a franchise is not treated as two different teams when its name changed.

In [ ]:
FEATURES = [
    "team1", "team2", "toss_winner", "toss_decision",
    "venue", "city", "season", "match_type"
]

TEAM_ALIASES = {
    "Royal Challengers Bangalore": "Royal Challengers Bengaluru",
    "Royal Challengers Bengaluru": "Royal Challengers Bengaluru",
    "Delhi Daredevils": "Delhi Capitals",
    "Delhi Capitals": "Delhi Capitals",
    "Kings XI Punjab": "Punjab Kings",
    "Punjab Kings": "Punjab Kings",
    "Rising Pune Supergiant": "Rising Pune Supergiant",
    "Rising Pune Supergiants": "Rising Pune Supergiant",
}

data = df[FEATURES + ["winner"]].dropna(subset=["winner"]).copy()

for col in ["team1", "team2", "toss_winner"]:
    data[col] = data[col].map(lambda x: TEAM_ALIASES.get(x, x))

data["winner"] = data["winner"].map(lambda x: TEAM_ALIASES.get(x, x))
for col in ["city", "venue", "toss_decision", "match_type"]:
    data[col] = data[col].fillna("Unknown")

# Only keep matches where the recorded winner is one of the two teams.
data = data[(data["winner"] == data["team1"]) | (data["winner"] == data["team2"])].copy()
data["team1_won"] = (data["winner"] == data["team1"]).astype(int)

print("Rows used:", len(data))
print("Team 1 wins:", data["team1_won"].sum())
print("Team 2 wins:", (data["team1_won"] == 0).sum())

## 5. Prepare training and testing data

In [ ]:
X = data[FEATURES]
y = data["team1_won"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

categorical_features = [
    "team1", "team2", "toss_winner", "toss_decision",
    "venue", "city", "match_type"
]
numeric_features = ["season"]

preprocessor = ColumnTransformer([
    ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_features),
    ("numeric", "passthrough", numeric_features)
])

X_train_encoded = preprocessor.fit_transform(X_train)
X_test_encoded = preprocessor.transform(X_test)

print("Training shape:", X_train_encoded.shape)
print("Testing shape:", X_test_encoded.shape)

## 6. Train Random Forest with GridSearchCV

In [ ]:
model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    class_weight="balanced",
    n_jobs=-1
)

param_grid = {
    "max_depth": [10, 15, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2]
}

grid = GridSearchCV(
    model, param_grid, cv=3, scoring="accuracy", n_jobs=-1
)
grid.fit(X_train_encoded, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV accuracy:", round(grid.best_score_, 4))

## 7. Evaluate the model

In [ ]:
best_model = grid.best_estimator_
predictions = best_model.predict(X_test_encoded)
probabilities = best_model.predict_proba(X_test_encoded)[:, 1]

accuracy = accuracy_score(y_test, predictions)
auc = roc_auc_score(y_test, probabilities)

print("Test accuracy:", round(accuracy, 4))
print("Test ROC-AUC:", round(auc, 4))
print("\nClassification report:")
print(classification_report(y_test, predictions, zero_division=0))

## 8. Confusion matrix

In [ ]:
cm = confusion_matrix(y_test, predictions)
display = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Team 2 wins", "Team 1 wins"])
display.plot()
plt.title("Confusion Matrix")
plt.show()

## 9. Conclusion

The final model is a Random Forest classifier that predicts whether Team 1 or Team 2 wins. The binary setup is useful for the web application because a prediction between RCB and CSK should only compare those two teams.

The model is an academic prediction system. Its result is affected by the limited features available in the dataset, so it should not be treated as a guaranteed match outcome.